In [27]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import re
import collections
import torch.nn.functional as F

# Text dataset

In [28]:
!wget -O timemachine.txt http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt

--2026-03-23 06:29:18--  http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt
Resolving d2l-data.s3-accelerate.amazonaws.com (d2l-data.s3-accelerate.amazonaws.com)... 3.163.164.229
Connecting to d2l-data.s3-accelerate.amazonaws.com (d2l-data.s3-accelerate.amazonaws.com)|3.163.164.229|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 178979 (175K) [text/plain]
Saving to: ‘timemachine.txt’

timemachine.txt     100%[===================>] 174.78K  --.-KB/s    in 0.02s   

2026-03-23 06:29:18 (9.49 MB/s) - ‘timemachine.txt’ saved [178979/178979]



In [29]:
def read_data():
    with open("timemachine.txt", 'r') as f:
        return f.read()

def process_text(text):
    return re.sub("[^A-Za-z]+", ' ', text).lower()

def tokenize(text):
    return list(text)

In [ ]:
class Vocab:
    def __init__(self, tokens, min_freq=0, reserved_tokens=[]):
        if isinstance(tokens[0], list):
            tokens = [t for line in tokens for t in line]
        counter = collections.Counter(tokens)
        self.freq_count = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        self.idx_to_tokens = list(sorted(set(["<unk>"] + reserved_tokens + [t for t,f in self.freq_count if f>=min_freq])))
        self.tokens_to_idx = {t:i for i,t in enumerate(self.idx_to_tokens)}

    def __len__(self): return len(self.idx_to_tokens)

    def __getitem__(self, tokens):
        if not isinstance(tokens, (list, tuple)):
            return self.tokens_to_idx.get(tokens, self.unk)
        return [self.__getitem__(t) for t in tokens]

    def to_tokens(self, indices):
        if hasattr(indices, "__len__") and len(indices) > 1:
            return [self.idx_to_tokens[i] for i in indices]
            
        return self.idx_to_tokens[indices]

    @property
    def unk(self): return self.tokens_to_idx["<unk>"]

vocab = Vocab(tokenize(process_text(read_data())))
len(vocab)

28

In [31]:
class TextData(torch.utils.data.Dataset):
    def __init__(self, text, num_steps):
        tokens = tokenize(text)
        self.vocab = Vocab(tokens=tokens)
        self.vocab_size = len(self.vocab)
        self.indices = [self.vocab[t] for t in tokens]
        self.array = torch.tensor([self.indices[i:i+num_steps+1] for i in range(len(self.indices) - num_steps)])
        self.X = self.array[:,:-1]
        self.y = self.array[:,1:]

text_data = TextData(process_text(read_data()), 32)

In [32]:
text_data.array

tensor([[ 8, 12,  3,  ..., 23,  1,  8],
        [12,  3,  1,  ...,  1,  8, 12],
        [ 3,  1,  8,  ...,  8, 12,  3],
        ...,
        [ 2,  8, 23,  ...,  1,  5,  7],
        [ 8, 23,  4,  ...,  5,  7, 27],
        [23,  4,  4,  ...,  7, 27,  1]])

In [33]:
text_data.X, text_data.y


(tensor([[ 8, 12,  3,  ...,  1, 23,  1],
         [12,  3,  1,  ..., 23,  1,  8],
         [ 3,  1,  8,  ...,  1,  8, 12],
         ...,
         [ 2,  8, 23,  ..., 14,  1,  5],
         [ 8, 23,  4,  ...,  1,  5,  7],
         [23,  4,  4,  ...,  5,  7, 27]]),
 tensor([[12,  3,  1,  ..., 23,  1,  8],
         [ 3,  1,  8,  ...,  1,  8, 12],
         [ 1,  8, 23,  ...,  8, 12,  3],
         ...,
         [ 8, 23,  4,  ...,  1,  5,  7],
         [23,  4,  4,  ...,  5,  7, 27],
         [ 4,  4,  1,  ...,  7, 27,  1]]))

In [34]:
num_train = 10000
num_val = 5000
def get_dataloader(text_data, shuffle, batch_size):
    indices = slice(0, num_train) if shuffle else slice(num_train, num_train+num_val)
    dataset = torch.utils.data.TensorDataset(text_data.X[indices], text_data.y[indices])
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_dataloader = get_dataloader(text_data, True, 1024)
eval_dataloader = get_dataloader(text_data, False, 1024)

# Model

In [35]:
class GRUfromScratch(nn.Module):
    def __init__(self,
                 num_inputs, # d
                 num_hiddens, # h
                 sigma=0.01):
        super(GRUfromScratch, self).__init__()
        init_weight = lambda *shape: nn.Parameter(torch.randn(*shape) * sigma)
        double = lambda: (init_weight(num_inputs+num_hiddens, num_hiddens),
                          nn.Parameter(torch.zeros(num_hiddens)))
        self.W_r, self.b_r = double() # reset gate
        self.W_z, self.b_z = double() # update gate
        self.W_h, self.b_h = double() # candidate hidden state

        self.num_hiddens = num_hiddens
        self.sigma = sigma

    def forward(self, inputs, H=None):
        if H is None:
            H = torch.zeros((inputs.shape[1], self.num_hiddens), device=inputs.device)
        outputs = []
        for X in inputs:
            R = torch.sigmoid(torch.matmul(torch.cat((X,H), dim=1), self.W_r) + self.b_r)
            Z = torch.sigmoid(torch.matmul(torch.cat((X,H), dim=1), self.W_z) + self.b_z)
            H_tilde = torch.tanh(torch.matmul(torch.cat((X,R*H), dim=1), self.W_h) + self.b_h)
            H = Z * H + (1 - Z) * H_tilde
            outputs.append(H)
        return outputs, H

In [40]:
class BiRNNFromScratch(nn.Module):
  def __init__(self, num_inputs, num_hiddens, sigma=0.01):
    super(BiRNNFromScratch, self).__init__()
    self.f_rnn = GRUfromScratch(num_inputs=num_inputs, num_hiddens=num_hiddens, sigma=sigma)
    self.b_rnn = GRUfromScratch(num_inputs=num_inputs, num_hiddens=num_hiddens, sigma=sigma)
    self.num_hiddens = 2 * num_hiddens
    self.sigma = sigma


  def forward(self, x, states):
    f_H, b_H = states if states is not None else (None, None)
    f_out, f_H = self.f_rnn(x, f_H)
    b_out, b_H = self.b_rnn(reversed(x), b_H)
    out = [torch.cat((f,b), dim=-1) for f,b in zip(f_out, reversed(b_out))]
    return out, (f_H, b_H)

In [41]:
class RNNLMfromScratch(nn.Module):
    def __init__(self, rnn, vocab_size):
        super(RNNLMfromScratch, self).__init__()
        self.W_hq = nn.Parameter(torch.randn(rnn.num_hiddens, vocab_size) * rnn.sigma)
        self.b_q = nn.Parameter(torch.zeros(vocab_size))
        self.vocab_size = vocab_size
        self.rnn = rnn

    def one_hot(self, x):
        return F.one_hot(x.T, self.vocab_size).type(torch.float32)

    def forward(self, x, state=None):
        embeds = self.one_hot(x)
        rnn_outputs, state = self.rnn(embeds, state)
        outputs = [torch.matmul(H, self.W_hq) + self.b_q for H in rnn_outputs]
        return torch.stack(outputs, dim=1), state

In [42]:
batch_size = 1024
num_inputs = 16 # feature dim
num_hiddens = 32
num_steps = 32
bi_rnn = BiRNNFromScratch(num_inputs=text_data.vocab_size, num_hiddens=num_hiddens)
x = torch.ones((batch_size, num_steps), dtype=torch.int64)
print(x.shape, F.one_hot(x, num_inputs).shape)
model = RNNLMfromScratch(bi_rnn, text_data.vocab_size)
out, _ = model(x)
print(out.shape)


torch.Size([1024, 32]) torch.Size([1024, 32, 16])
torch.Size([1024, 32, 28])


In [ ]:
def grad_clip(clip_val, model):
    params = [p for p in model.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum((p.grad**2)) for p in params))
    if norm > clip_val:
        for p in params:
            p.grad[:] *= clip_val / norm

In [44]:
def train_deep_rnn(model, max_epochs, clip_val, lr, train_dataloader, eval_dataloader, report_every):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    metrics = {"train_loss": [], "train_ppl": [], "eval_loss": [], "eval_ppl": []}
    criterion = nn.CrossEntropyLoss()

    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr)

    for epoch in range(max_epochs):
        model.train()
        num_instances = 0
        epoch_loss = 0.
        epoch_ppl = 0.
        for step, batch in enumerate(train_dataloader):
            optimizer.zero_grad()
            batch = [a.to(device) for a in batch]
            bs = batch[-1].size(0)
            logits, _ = model(*batch[:-1])
            logits = logits.reshape(-1, logits.shape[-1])
            loss = criterion(logits, batch[-1].reshape(-1))
            loss.backward()
            grad_clip(clip_val, model)
            optimizer.step()
            epoch_loss += loss.item() * bs
            epoch_ppl += torch.exp(loss).item() * bs
            num_instances += bs

        epoch_loss /= num_instances
        epoch_ppl /= num_instances
        metrics["train_loss"].append(epoch_loss)
        metrics["train_ppl"].append(epoch_ppl)
        if epoch % report_every==0:
            print(f"[{epoch}/{max_epochs}] train_loss: {epoch_loss:.5f}, train_ppl: {epoch_ppl:.5f}")

        model.eval()
        num_instances = 0
        epoch_loss = 0.
        epoch_ppl = 0.
        for step, batch in enumerate(eval_dataloader):
            batch = [a.to(device) for a in batch]
            bs = batch[-1].size(0)
            with torch.no_grad():
                logits, _ = model(*batch[:-1])
                logits = logits.reshape(-1, logits.shape[-1])
                loss = criterion(logits, batch[-1].reshape(-1))
                epoch_loss += loss.item() * bs
                epoch_ppl += torch.exp(loss).item() * bs
            num_instances += bs

        epoch_loss /= num_instances
        epoch_ppl /= num_instances
        metrics["eval_loss"].append(epoch_loss)
        metrics["eval_ppl"].append(epoch_ppl)
        if epoch % report_every==0:
            print(f"[{epoch}/{max_epochs}] eval_loss: {epoch_loss:.5f}, eval_ppl: {epoch_ppl:.5f}")

    return metrics


In [45]:
# plt.plot(metrics["train_ppl"], label="train")
# plt.plot(metrics["eval_ppl"], label="eval")
# plt.legend(); plt.show()

In [46]:
class BiRNN(nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super(BiRNN, self).__init__()
        self.rnn = nn.GRU(input_size=num_inputs, hidden_size=num_hiddens, bidirectional=True)
        self.num_hiddens = 2 * num_hiddens

    def forward(self, x, H_C=None):
        return self.rnn(x, H_C)

In [47]:
class RNNLM(nn.Module):
    def __init__(self, rnn, vocab_size):
        super(RNNLM, self).__init__()
        self.linear = nn.Linear(in_features=rnn.num_hiddens, out_features=vocab_size)
        self.vocab_size = vocab_size
        self.rnn = rnn

    def one_hot(self, x):
        return F.one_hot(x.T, self.vocab_size).type(torch.float32)

    def forward(self, x, state=None):
        embeds = self.one_hot(x)
        rnn_outputs, state = self.rnn(embeds, state)
        outputs = self.linear(rnn_outputs).swapaxes(0,1)
        return outputs, state

In [48]:
# plt.plot(metrics["train_ppl"], label="train")
# plt.plot(metrics["eval_ppl"], label="eval")
# plt.legend(); plt.show()

In [49]:
# predict("it has", 20, text_data.vocab, model, device=device)